In [ ]:
# X6ACQ — pulls the code from GitHub so a run is reproducible from a commit SHA
import os, subprocess, sys, shutil, time
REPO = "https://github.com/Ahmadrezanourozii/Automatic-detection-of-diabetic-retinopathy-and-grading-of-diabetic-macular-edema-using-CNN.git"
COMMIT = "d3c42f71dbf02979585d70ea7aa51805c1487624"
WORK = "/kaggle/working/repo"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO, WORK], check=True)
if COMMIT and COMMIT != "HEAD":
    subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
sha = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"]).decode().strip()
print("CODE COMMIT", sha)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--pretty=%s"]).decode().strip())


In [ ]:
import os
for d in sorted(os.listdir("/kaggle/input")):
    n = sum(len(f) for _, _, f in os.walk(f"/kaggle/input/{d}"))
    print(f"{d:55s} {n:7d} files")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
import subprocess, sys, os
OUT = "/kaggle/working/X6ACQ"
os.makedirs(OUT, exist_ok=True)
# post-hoc shortcut check: acquisition statistics of every image, with its grade
cmd = [sys.executable, "-u", "/kaggle/working/repo/src/paper_degrade.py",
       "--datasets", "/kaggle/input", "--out", OUT,
       "--per-image", "SUSTech,APTOS,DeepDRiD,DDR,IDRiD,Messidor-2"]
print(" ".join(cmd), flush=True)
with open(f"{OUT}/run.log", "w") as f:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd="/kaggle/working/repo")
    for line in p.stdout:
        print(line, end="", flush=True); f.write(line); f.flush()
    p.wait()
print("exit=", p.returncode)
if p.returncode:
    raise SystemExit(p.returncode)
